# Grid based Bidirectional A* (Two-Side Search)

Randomly generate obstacles, start and goal point
searching path from start and end simultaneously

---- 

- conda env : [ai_robotics](../../README.md#setup-a-conda-environment)

---

### Ref
- https://github.com/AtsushiSakai/PythonRobotics/
- https://github.com/AtsushiSakai/PythonRobotics/blob/master/PathPlanning/AStar/a_star_searching_from_two_side.py
- https://en.wikipedia.org/wiki/Bidirectional_search

### Imports and Configuration

In [9]:
"""
A* Algorithm with FuncAnimation Visualization
Author: Adapted from Weicent by ChatGPT (GPT-5)
"""

import numpy as np
import matplotlib.pyplot as plt
import math
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

### Path Planner

In [10]:
# ==============================================
# Configuration
# ==============================================
show_animation = True
interval_ms = 50  # frame update speed (milliseconds)

# ==============================================
# Node Class Definition
# ==============================================
class Node:
    def __init__(self, G=0, H=0, coordinate=None, parent=None):
        self.G = G
        self.H = H
        self.F = G + H
        self.parent = parent
        self.coordinate = coordinate

    def reset_f(self):
        self.F = self.G + self.H


# ==============================================
# Utility Functions
# ==============================================
def hcost(node_coordinate, goal):
    dx = abs(node_coordinate[0] - goal[0])
    dy = abs(node_coordinate[1] - goal[1])
    return dx + dy


def gcost(fixed_node, update_node_coordinate):
    dx = abs(fixed_node.coordinate[0] - update_node_coordinate[0])
    dy = abs(fixed_node.coordinate[1] - update_node_coordinate[1])
    return fixed_node.G + math.hypot(dx, dy)


def boundary_and_obstacles(start, goal, top_vertex, bottom_vertex, obs_number):
    ay = list(range(bottom_vertex[1], top_vertex[1]))
    ax = [bottom_vertex[0]] * len(ay)
    cy = ay
    cx = [top_vertex[0]] * len(cy)
    bx = list(range(bottom_vertex[0] + 1, top_vertex[0]))
    by = [bottom_vertex[1]] * len(bx)
    dx = [bottom_vertex[0]] + bx + [top_vertex[0]]
    dy = [top_vertex[1]] * len(dx)

    ob_x = np.random.randint(bottom_vertex[0] + 1, top_vertex[0], obs_number).tolist()
    ob_y = np.random.randint(bottom_vertex[1] + 1, top_vertex[1], obs_number).tolist()
    obstacle = np.vstack((ob_x, ob_y)).T.tolist()
    obstacle = [coor for coor in obstacle if coor != start and coor != goal]
    obs_array = np.array(obstacle)
    bound = np.vstack((ax + bx + cx + dx, ay + by + cy + dy)).T
    return np.vstack((bound, obs_array)), obstacle


def find_neighbor(node, ob, closed):
    ob_set = set(map(tuple, ob.tolist()))
    neighbor_set = set()
    for x in range(node.coordinate[0] - 1, node.coordinate[0] + 2):
        for y in range(node.coordinate[1] - 1, node.coordinate[1] + 2):
            coord = (x, y)
            if coord not in ob_set and coord != tuple(node.coordinate):
                neighbor_set.add(coord)
    closed_set = set(map(tuple, closed))
    return list(neighbor_set - closed_set)


def find_node_index(coordinate, node_list):
    for i, node in enumerate(node_list):
        if node.coordinate == coordinate:
            return i
    return -1


def node_to_coordinate(node_list):
    return [node.coordinate for node in node_list]


def check_node_coincide(cl1, cl2):
    coords1 = node_to_coordinate(cl1)
    coords2 = node_to_coordinate(cl2)
    intersect = [c for c in coords1 if c in coords2]
    return intersect


def get_path(org_list, goal_list, coordinate):
    path_org, path_goal = [], []
    i = find_node_index(coordinate, org_list)
    node = org_list[i]
    while node.parent:
        path_org.append(node.coordinate)
        node = node.parent
    path_org.append(org_list[0].coordinate)
    i = find_node_index(coordinate, goal_list)
    node = goal_list[i]
    while node.parent:
        path_goal.append(node.coordinate)
        node = node.parent
    path_goal.append(goal_list[0].coordinate)
    path_org.reverse()
    return np.array(path_org + path_goal)


# ==============================================
# A* Bidirectional Search Logic with Generator
# ==============================================
def bidirectional_astar(start, end, bound, obstacle):
    origin = Node(coordinate=start, H=hcost(start, end))
    goal = Node(coordinate=end, H=hcost(end, start))
    origin_open, origin_close = [origin], []
    goal_open, goal_close = [goal], []
    target_goal, flag = end, 0

    while True:
        if not origin_open or not goal_open:
            flag = 1
            yield origin_close, goal_close, flag, None
            return

        # Step 1: From start
        node = min(origin_open, key=lambda x: x.F)
        origin_open.remove(node)
        origin_close.append(node)
        for ncoord in find_neighbor(node, bound, node_to_coordinate(origin_close)):
            new_g = gcost(node, ncoord)
            new_node = Node(G=new_g, H=hcost(ncoord, target_goal), coordinate=ncoord, parent=node)
            origin_open.append(new_node)

        # Step 2: From goal
        node_g = min(goal_open, key=lambda x: x.F)
        goal_open.remove(node_g)
        goal_close.append(node_g)
        for ncoord in find_neighbor(node_g, bound, node_to_coordinate(goal_close)):
            new_g = gcost(node_g, ncoord)
            new_node = Node(G=new_g, H=hcost(ncoord, start), coordinate=ncoord, parent=node_g)
            goal_open.append(new_node)

        intersect = check_node_coincide(origin_close, goal_close)
        if intersect:
            path = get_path(origin_close, goal_close, intersect[0])
            yield origin_close, goal_close, 0, path
            return

        yield origin_close, goal_close, flag, None







In [11]:
# ==============================================
# Visualization with FuncAnimation
# ==============================================
def run_animation(obstacle_number=800):
    top_vertex, bottom_vertex = [60, 60], [0, 0]
    start = [np.random.randint(1, 59), np.random.randint(1, 59)]
    end = [np.random.randint(1, 59), np.random.randint(1, 59)]
    bound, obstacle = boundary_and_obstacles(start, end, top_vertex, bottom_vertex, obstacle_number)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(bound[:, 0], bound[:, 1], 'sk', markersize=3)
    ax.plot(start[0], start[1], '^b', label='Start', markersize=10)
    ax.plot(end[0], end[1], '*r', label='Goal', markersize=10)
    ax.legend()
    ax.set_xlim(bottom_vertex[0], top_vertex[0])
    ax.set_ylim(bottom_vertex[1], top_vertex[1])
    ax.set_aspect('equal')

    generator = bidirectional_astar(start, end, bound, obstacle)
    search_start, = ax.plot([], [], 'oy', label="Search Start")
    search_goal, = ax.plot([], [], 'og', label="Search Goal")
    path_line, = ax.plot([], [], '-r', linewidth=2)

    def update(frame):
        org_closed, goal_closed, flag, path = frame
        org_coords = np.array(node_to_coordinate(org_closed))
        goal_coords = np.array(node_to_coordinate(goal_closed))
        if len(org_coords) > 0:
            search_start.set_data(org_coords[:, 0], org_coords[:, 1])
        if len(goal_coords) > 0:
            search_goal.set_data(goal_coords[:, 0], goal_coords[:, 1])
        if path is not None:
            path_line.set_data(path[:, 0], path[:, 1])
        return search_start, search_goal, path_line

    ani = FuncAnimation(fig, update, frames=generator, interval=interval_ms, repeat=False)
    plt.close(fig)  # prevent duplicate static plot
    return ani

In [12]:
# ==============================================
# Run
# ==============================================
ani = run_animation(obstacle_number=1000)
HTML(ani.to_html5_video())

/var/folders/t1/1hj3rh7x4w30zbxz0z0_x3_r0000gn/T/ipykernel_73114/603710748.py:36: UserWarning: frames=<generator object bidirectional_astar at 0x10f9a5150> which we can infer the length of, did not pass an explicit *save_count* and passed cache_frame_data=True.  To avoid a possibly unbounded cache, frame data caching has been disabled. To suppress this warning either pass `cache_frame_data=False` or `save_count=MAX_FRAMES`.
  ani = FuncAnimation(fig, update, frames=generator, interval=interval_ms, repeat=False)
